# Gradient Descent Approach with Polynomial Roots

In [47]:
# Core scientific stack
import numpy as np
import sympy as sp
import networkx as nx

# Modeling utilities
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from skorch import NeuralNetClassifier, NeuralNetRegressor
    print("PyTorch available")
except Exception as e:
    print("PyTorch not available; install it to train the baseline.")
    torch = None

from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash
from tqdm.notebook import tqdm # Switch this to regular tqdm when running in a script

# Optional plotting for quick visual checks
import matplotlib.pyplot as plt

np.random.seed(42)

PyTorch available


## 1) One-Vertex Isospectral Reduction

In [48]:
lam = sp.Symbol('lam')

def one_vertex_reduction(A: np.ndarray, keep_vertex: int, symbol=lam):
    """
    Compute the one-vertex isospectral reduction R_i(lam) for a numeric matrix A.
    Returns a SymPy expression (typically rational in lam).
    """
    n = A.shape[0]
    all_idx = list(range(n))
    rest = [j for j in all_idx if j != keep_vertex]

    A_sym = sp.Matrix(A)

    # Degenerate 1x1 case: reduction is just the diagonal entry
    if len(rest) == 0:
        return sp.simplify(A_sym[keep_vertex, keep_vertex])
    
    A_ii = sp.Matrix([[A_sym[keep_vertex, keep_vertex]]])
    A_iR = A_sym.extract([keep_vertex], rest)
    A_RR = A_sym.extract(rest, rest)
    A_Ri = A_sym.extract(rest, [keep_vertex])

    I = sp.eye(len(rest))
    reduced = A_ii - A_iR * (A_RR - symbol * I).inv() * A_Ri
    return sp.together(sp.simplify(reduced[0, 0]))

## 2) Prepare Features from Polynomial Roots

In [49]:
def split_roots(expr, symbol=lam, imag_tol=1e-8, keep_complex=True):
    num, den = sp.fraction(sp.together(expr))
    num_poly = sp.Poly(sp.expand(num), symbol)
    den_poly = sp.Poly(sp.expand(den), symbol)

    def get_roots(poly):
        if poly.degree() <= 0:
            return np.array([], dtype=complex)
        vals = np.array([complex(r) for r in sp.nroots(poly)], dtype=complex)
        if not keep_complex:
            vals = np.array([z.real for z in vals if abs(z.imag) < imag_tol], dtype=float)
        return vals
    
    return {"num_roots": get_roots(num_poly), "den_roots": get_roots(den_poly)}

In [50]:
def roots_to_features(root_dict, k_num=20, k_den=20):
    """Encode numerator/denominator roots into a fixed-width real vector"""
    num_roots = root_dict["num_roots"]
    den_roots = root_dict["den_roots"]

    def pack(roots, k):
        roots = np.array(sorted(roots, key=lambda z: abs(z), reverse=True), dtype=complex)
        real_part = np.zeros(k, dtype=float)
        imag_part = np.zeros(k, dtype=float)
        magnitude = np.zeros(k, dtype=float)

        # TODO: Remove this min and pack with NaNs?
        m = min(k, len(roots)) 
        if m > 0:
            real_part[:m] = np.real(roots[:m])
            imag_part[:m] = np.imag(roots[:m])
            magnitude[:m] = np.abs(roots[:m])

        return np.concatenate([real_part, imag_part, magnitude])
    
    return np.concatenate([pack(num_roots, k_num), pack(den_roots, k_den)])

FEATURE_DIM = roots_to_features({"num_roots": np.array([], dtype=complex), "den_roots": np.array([], dtype=complex)}).shape[0]
print(f"Feature dimension per sample: {FEATURE_DIM}")

Feature dimension per sample: 120


## 3) Build a train/test split dataset from simple connected graphs

In [51]:
def random_unique_connected_graphs(min_nodes=6, max_nodes=20, max_graphs=100, seed=None):
    """
    Generate random, unique connected graphs in a node range.
    """
    if seed is not None:
        np.random.seed(seed)

    selected = []
    seen = set()
    attempts = 0
    max_attempts = max_graphs * 50 # prevent infinite loops

    while len(selected) < max_graphs and attempts < max_attempts:
        attempts += 1

        n = np.random.randint(min_nodes, max_nodes)
        G = nx.gnp_random_graph(n,np.random.rand())

        if G.number_of_edges() == 0:
            continue
        if not nx.is_connected(G):
            continue

        # Canonical representation for uniqueness
        h = weisfeiler_lehman_graph_hash(G)

        if h in seen:
            continue

        seen.add(h)
        selected.append((len(selected), nx.convert_node_labels_to_integers(G)))

    return selected

In [ ]:
def generate_dataset(min_nodes=6, max_nodes=20, max_graphs=100, seed=None):
    """
    Generate the dataset with classes of interest
    """
    graphs = random_unique_connected_graphs(min_nodes, max_nodes, max_graphs, seed)

    total_vertices = sum(G.number_of_nodes() for _, G in graphs)
    X_list = []
    samples_graph_label = []
    samples_edge_class = []
    samples_node_class = []
    samples_vertex = []
    samples_roots = []
    label_to_graph = {}
    reduction_dict = {}
    counts = {}

    with tqdm(total=total_vertices, desc="Processing vertices") as pbar:
        for graph_label, G in graphs:
            label_to_graph[graph_label] = G
            A = nx.to_numpy_array(G, dtype=int)
            edge_class = G.number_of_edges()
            node_class = G.number_of_nodes()

            # One training sample per (graph, vertex)
            for v in range(node_class):
                expr = one_vertex_reduction(A, v, symbol=lam)
                # expr_simplified = sp.together(sp.simplify(expr))
                # key = sp.srepr(expr_simplified)

                # if key not in counts:
                #     counts[key] = {}

                # if node_class not in counts[key]:
                #     counts[key][node_class] = 1
                # else:
                #     counts[key][node_class] += 1
                # reduction_dict[key] = expr_simplified  # store pretty version
                roots_dict = split_roots(expr, symbol=lam)
                feat = roots_to_features(roots_dict, k_num=20, k_den=20)

                X_list.append(feat)
                samples_graph_label.append(graph_label)
                samples_edge_class.append(edge_class)
                samples_node_class.append(node_class)
                samples_vertex.append(v)
                samples_roots.append(roots_dict)

                pbar.update(1)

        X = np.array(X_list, dtype=float)
        y = np.array(samples_node_class)
        groups = np.array(samples_graph_label)

    return [X, y, groups, samples_graph_label, samples_edge_class, samples_node_class,
            samples_vertex, samples_roots, label_to_graph, reduction_dict, counts]

In [53]:
def train_test_split(X, y, groups, samples_graph_label, samples_edge_class, samples_node_class,
            samples_vertex, samples_roots, label_to_graph, reduction_dict, counts):

    # Grouped split by graph ID: vertices from the same graph stay in one split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    v_train = np.array(samples_vertex)[train_idx]
    v_test = np.array(samples_vertex)[test_idx]
    graph_id_train = np.array(samples_graph_label)[train_idx]
    graph_id_test = np.array(samples_graph_label)[test_idx]
    n_train = np.array(samples_node_class)[train_idx]
    n_test = np.array(samples_node_class)[test_idx]

    print(f"Train samples: {len(train_idx)} | Test samples: {len(test_idx)}")
    print(f"Train graph IDs: {len(np.unique(graph_id_train))} | Test graph IDs: {len(np.unique(graph_id_test))}")
    print(f"Train classes: {sorted(np.unique(y_train).tolist())}")
    print(f"Test classes:  {sorted(np.unique(y_test).tolist())}")
    print(f"Train node-counts: {sorted(np.unique(n_train).tolist())}")
    print(f"Test node-counts:  {sorted(np.unique(n_test).tolist())}")

    labels_eval = np.unique(y_test)

    return [X, y, groups, samples_graph_label, samples_edge_class, samples_node_class, samples_vertex, samples_roots, label_to_graph, reduction_dict, counts, train_idx, test_idx, X_train, X_test, y_train, y_test, v_train, v_test, graph_id_train, graph_id_test, n_train, n_test, labels_eval]

## 4a) Train node-class models, then score by root-consistent unfolding (interpolation)

Goal: Attempt an ensemble of models with the first model predicting the number of nodes

* Logistic Regression
* Random Forest
* MLP Neural Net

In [54]:
X, y, groups, samples_graph_label, samples_edge_class, samples_node_class, samples_vertex, samples_roots, label_to_graph, reduction_dict, counts, train_idx, test_idx, X_train, X_test, y_train, y_test, v_train, v_test, graph_id_train, graph_id_test, n_train, n_test, labels_eval = train_test_split(*generate_dataset(min_nodes=6, max_nodes=10, max_graphs=200))

/home/dallin/Documents/Classes/Graph_Theory/isospectral_pst/unfolding/unfolding_ml/.venv/lib/python3.12/site-packages/networkx/algorithms/graph_hashing.py:211: UserWarning: The hashes produced for graphs without node or edge attributeschanged in v3.5 due to a bugfix (see documentation).
  node_labels = _init_node_labels(G, edge_attr, node_attr)


Processing vertices:   0%|          | 0/1564 [00:00<?, ?it/s]

Train samples: 1182 | Test samples: 382
Train graph IDs: 150 | Test graph IDs: 50
Train classes: [6, 7, 8, 9]
Test classes:  [6, 7, 8, 9]
Train node-counts: [6, 7, 8, 9]
Test node-counts:  [6, 7, 8, 9]


In [55]:
logit_model = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0.0),
    StandardScaler(),
    LogisticRegression(max_iter=5000, class_weight='balanced')
)

rf_model = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0.0),
    RandomForestClassifier(
        n_estimators=500,
        max_depth=18,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1,
    ),
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=512, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)
    
torch_model = NeuralNetClassifier(
    MLP,
    module__in_dim=X_train.shape[1],
    module__hidden=512,
    module__out_dim=len(np.unique(y_train)),
    max_epochs=30,
    lr=1e-3,
    batch_size=64,
    optimizer=torch.optim.Adam,
    criterion=nn.CrossEntropyLoss,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=0,
)

to_float32 = FunctionTransformer(lambda X: X.astype(np.float32))
nn_model = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0.0),
    StandardScaler(),
    to_float32,
    torch_model
)

# Pytorch expects labels in {0, 1, 2, ...}
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

results = {}
for name, model in [
    ("LogisticRegression", logit_model), 
    ("RandomForest", rf_model),
    ("NeuralNet", nn_model)
    ]:
    model.fit(X_train, y_train_enc)
    pred_local = model.predict(X_test)
    # pred_original = le.inverse_transform(pred_local) <- Convert labels back to node classes
    results[name] = {
        'model': model,
        'pred': pred_local,
        'acc': accuracy_score(y_test_enc, pred_local),
        'bacc': recall_score(y_test_enc, pred_local, average='macro', labels=np.unique(y_test_enc), zero_division=0),
        'macro_f1': f1_score(y_test_enc, pred_local, average='macro', labels=np.unique(y_test_enc), zero_division=0),
    }
    print(f"{name:18s} | Acc={results[name]['acc']:.4f} | BAcc={results[name]['bacc']:.4f} | Macro-F1={results[name]['macro_f1']:.4f}")

best_model_name = max(results, key=lambda k: results[k]['bacc'])
best_model = results[best_model_name]['model']
pred = results[best_model_name]['pred']

print(f"\nBest by balanced accuracy: {best_model_name}")
print("\nClassification report (best model):")
print(classification_report(y_test_enc, pred, labels=np.unique(y_test_enc), zero_division=0))

LogisticRegression | Acc=0.8351 | BAcc=0.8299 | Macro-F1=0.8254
RandomForest       | Acc=0.8063 | BAcc=0.8042 | Macro-F1=0.8021
NeuralNet          | Acc=0.8429 | BAcc=0.8308 | Macro-F1=0.8316

Best by balanced accuracy: NeuralNet

Classification report (best model):
              precision    recall  f1-score   support

           0       0.81      0.78      0.80        60
           1       0.76      0.80      0.78        84
           2       0.88      0.82      0.85       112
           3       0.89      0.92      0.90       126

    accuracy                           0.84       382
   macro avg       0.83      0.83      0.83       382
weighted avg       0.84      0.84      0.84       382



## 4b) Train node-class models, then score by root-consistent unfolding (extrapolation via regression)

Goal: Attempt an ensemble of models with the first model predicting the number of nodes

* Logistic Regression
* Random Forest
* MLP Neural Net

In [56]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=512, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)
    
torch_model = NeuralNetRegressor(
    MLP,
    module__in_dim=X_train.shape[1],
    module__out_dim=1,
    max_epochs=30,
    lr=1e-3,
    batch_size=64,
    optimizer=torch.optim.Adam,
    criterion=nn.MSELoss,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=0,
)

to_float32 = FunctionTransformer(lambda X: X.astype(np.float32))
nn_model = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0.0),
    StandardScaler(),
    to_float32,
    torch_model
)

results = {}
for name, model in [
    ("NeuralNet", nn_model)
    ]:
    y_train_prime = y_train.astype(np.float32).reshape(-1, 1)
    y_test_prime  = y_test.astype(np.float32).reshape(-1, 1)
    model.fit(X_train, y_train_prime)
    pred = model.predict(X_test)
    # pred_original = le.inverse_transform(pred_local) <- Convert labels back to node classes
    results[name] = {
        'model': model,
        'pred': pred,
        "MAE": mean_absolute_error(y_test, pred),
    }
    print(f"{name:2s} | MAE={results[name]['MAE']:.4f}")

best_model_name = min(results, key=lambda k: results[k]['MAE'])
best_model = results[best_model_name]['model']
pred = results[best_model_name]['pred']
pred_class = np.round(pred).astype(int)
accuracy = np.mean(pred_class.flatten() == y_test.flatten())

print(f"\nBest by MAE: {best_model_name}")
print(y_test_prime - pred)
print(np.unique(pred_class))
print(f"Accuracy: {accuracy}")

NeuralNet | MAE=0.4711

Best by MAE: NeuralNet
[[ 0.14254951]
 [ 0.04934311]
 [-0.08172655]
 [-0.02873993]
 [ 0.1867094 ]
 [ 0.4472413 ]
 [ 0.284338  ]
 [ 0.03443146]
 [-0.23921013]
 [ 0.03443146]
 [-0.3057394 ]
 [-0.23921013]
 [ 0.06539536]
 [-0.38345146]
 [-0.38345146]
 [-0.3057394 ]
 [-0.8962259 ]
 [-0.8962259 ]
 [-0.6159873 ]
 [-0.8962259 ]
 [-0.26342392]
 [-0.6159873 ]
 [-0.26342392]
 [-0.49154282]
 [-0.05374527]
 [-0.05374527]
 [-0.49154282]
 [-0.05374527]
 [-0.05374527]
 [-0.49154282]
 [-0.49154282]
 [-0.66232777]
 [-0.25519085]
 [-0.81999254]
 [-0.10726261]
 [-0.81999254]
 [-0.66232777]
 [-0.34766293]
 [-0.06936455]
 [-0.18566418]
 [-0.05741692]
 [-0.34766293]
 [-0.05741692]
 [ 0.02370834]
 [-0.34766293]
 [-0.73417854]
 [-1.2178698 ]
 [-1.1257467 ]
 [ 0.01432896]
 [-2.0511742 ]
 [-1.4588337 ]
 [-0.7758541 ]
 [ 0.19095802]
 [-0.47346878]
 [-0.33010197]
 [-0.5241256 ]
 [-0.59620094]
 [-0.3429289 ]
 [-0.59620094]
 [-0.5241256 ]
 [-0.38333416]
 [-0.38333416]
 [-0.22825193]
 [-0.228